# HW02 — MLflow Experiment Tracking

In HW01, you built a versioned feature dataset for the Airbnb listing availability problem.

In this notebook, you will train several model versions and track them in MLflow.

The goal is not only to get a high score. The goal is to make every experiment reproducible:

- which dataset version was used
- which features were used
- which model was trained
- which parameters were used
- which metrics were produced
- which artifacts were saved
- which run should be considered the final candidate

MLflow server:

```text
http://185.50.38.163:33014
```

Use your assigned MLflow username/password and your assigned experiment name from the credentials sheet.

## Required output

By the end of this notebook, you must have:

1. At least **5 MLflow runs**.
2. At least **3 different experiment types**:
   - one intentionally leaky run
   - one baseline run
   - at least one clean real model
3. Logged parameters, metrics, tags, artifacts, and an sklearn Pipeline model.
4. A run comparison table.
5. One selected final candidate run.
6. A short explanation of why that run was selected.

Do not use future/label columns in your final clean model.

In [1]:
# If needed, install these in your local environment first:
# pip install pandas numpy scikit-learn matplotlib mlflow pyarrow

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42

## 1. Configure MLflow

Fill in your assigned MLflow credentials.

Important:

- `MLFLOW_TRACKING_URI` is the shared MLflow server.
- `MLFLOW_USERNAME` and `MLFLOW_PASSWORD` are **not** your database credentials.
- `EXPERIMENT_NAME` must be your own assigned experiment name, for example:

```text
qbc12_hw02_student_nazanin_hesari
```

Do not use someone else's experiment name.

In [ ]:
MLFLOW_TRACKING_URI = ""

# TODO: replace these with your assigned MLflow credentials.
MLFLOW_USERNAME = ""
MLFLOW_PASSWORD = ""
EXPERIMENT_NAME = ""

if MLFLOW_USERNAME == "student_your_username" or MLFLOW_PASSWORD == "your_mlflow_password":
    raise ValueError("Replace MLFLOW_USERNAME, MLFLOW_PASSWORD, and EXPERIMENT_NAME with your assigned values.")

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)

## 2. Load the HW01 dataset

Use the cleaned dataset produced by HW01.

Expected files:

```text
data/features/listing_availability_features_v1_audit_cleaned.csv
data/features/listing_availability_features_v1_audit_cleaned.parquet
data/features/listing_availability_features_v1_audit_cleaned_metadata.json
```

You may use CSV or Parquet. Parquet is preferred if available.

In [3]:
DATASET_VERSION = "V1_student"

FEATURE_DIR = Path("data/features")

parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"

# TODO: load the dataset.
# Prefer Parquet if it exists, otherwise use CSV.
if parquet_path:
    feature_df =  pd.read_parquet(parquet_path)
    
elif csv_path:
    feature_df = pd.read_csv(csv_path)
    
else:
    raise NotImplementedError("Load feature_df from parquet_path or csv_path.")

# TODO: load metadata if metadata_path exists.
metadata = {}

if metadata_path:
    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

print(feature_df.shape)
feature_df.head()

(9383, 32)


,listing_id,neighbourhood_name,is_superhost,room_type,property_type,accommodates,bedrooms,beds,bathrooms,listing_price,...,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,cutoff_date,dataset_version
0,27886,Centrum-West,1,Private room,Private room in houseboat,2,1.0,1.0,1.5,132.0,...,1,0.033333,3.000000,30.0,30,0,0.000000,1.0,2026-01-01,v1_student
1,28871,Centrum-West,1,Private room,Private room in rental unit,2,1.0,1.0,0.2,89.0,...,6,0.200000,1.933333,730.0,30,1,0.033333,1.0,2026-01-01,v1_student
2,29051,Centrum-Oost,1,Private room,Private room in condo,2,1.0,1.0,0.2,61.0,...,2,0.066667,2.000000,730.0,30,0,0.000000,1.0,2026-01-01,v1_student
3,44391,Centrum-Oost,0,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,NaN,...,0,0.000000,3.000000,730.0,30,0,0.000000,1.0,2026-01-01,v1_student
4,48373,Buitenveldert - Zuidas,0,Entire home/apt,Entire rental unit,4,2.0,NaN,1.5,NaN,...,0,0.000000,3.000000,1125.0,30,0,0.000000,1.0,2026-01-01,v1_student


## 3. Define target and forbidden columns

The target is:

```text
high_demand_proxy
```

The following columns must **not** be used as clean model inputs:

```text
listing_id
cutoff_date
dataset_version
future_calendar_days_observed_30d
future_available_days_30d
future_available_rate_30d
high_demand_proxy
```

Why?

- `high_demand_proxy` is the label.
- `future_*` columns are from the label window.
- `listing_id`, `cutoff_date`, and `dataset_version` are audit/entity fields, not predictive features.

You will intentionally use one future column in the **leaky run only** to show what leakage looks like. Your final model must be clean.

In [4]:
TARGET_COL = "high_demand_proxy"

FORBIDDEN_MODEL_COLUMNS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

# TODO: check that TARGET_COL exists.
assert TARGET_COL in feature_df.columns, f" target column {TARGET_COL} does not exist"
# TODO: create y.
y = feature_df[TARGET_COL]
# TODO: create clean feature list by excluding FORBIDDEN_MODEL_COLUMNS.
clean_feature_cols = feature_df.columns.difference(FORBIDDEN_MODEL_COLUMNS)
# TODO: create X_clean.
X_clean = feature_df[clean_feature_cols]

if y.empty or len(clean_feature_cols) == 0 or  X_clean.empty:
    raise NotImplementedError("Create y, clean_feature_cols, and X_clean.")

print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())

print("Clean feature count:", len(clean_feature_cols))
print(clean_feature_cols)

Target distribution:
high_demand_proxy
0.0    0.311308
1.0    0.688692
Name: proportion, dtype: float64
Clean feature count: 25
Index(['accommodates', 'available_days_last_30d', 'available_days_last_90d',
       'available_rate_last_30d', 'available_rate_last_90d',
       'avg_comment_len_before_cutoff', 'avg_maximum_nights_calendar_last_30d',
       'avg_maximum_nights_calendar_last_90d',
       'avg_minimum_nights_calendar_last_30d',
       'avg_minimum_nights_calendar_last_90d', 'bathrooms', 'bedrooms', 'beds',
       'days_since_last_review', 'instant_bookable', 'is_superhost',
       'listing_price', 'max_comment_len_before_cutoff', 'maximum_nights',
       'minimum_nights', 'neighbourhood_name', 'property_type', 'room_type',
       'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff'],
      dtype='object')


## 4. Create one intentionally leaky feature set

This run is supposed to be wrong.

Create `X_leaky` by allowing `future_available_rate_30d` into the features.

The point is to show that a model can look excellent for the wrong reason. Log this run with:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
```

Do not select this run as your final model.

In [5]:
LEAKAGE_COLUMN = "future_available_rate_30d"

# TODO: create leaky_feature_cols.
# It should include the clean features plus LEAKAGE_COLUMN.
# It must still exclude the target itself.

leaky_feature_cols = clean_feature_cols.union([LEAKAGE_COLUMN])
X_leaky = feature_df[leaky_feature_cols]

assert LEAKAGE_COLUMN in feature_df.columns, "Leakage column missing"
assert TARGET_COL not in clean_feature_cols, "Target leaked into features"

if len(leaky_feature_cols) == 0 or  X_leaky.empty:
    raise NotImplementedError("Create leaky_feature_cols and X_leaky.")

print("Leaky feature count:", len(leaky_feature_cols))
print("Leakage column included:", LEAKAGE_COLUMN in leaky_feature_cols)

Leaky feature count: 26
Leakage column included: True


## 5. Train/test split

Use a stratified split.

Why stratified?

The target is not perfectly balanced, so the train and test sets should preserve the class ratio.

In [6]:
# TODO: split X_clean and y.
# Use test_size=0.20, random_state=42, stratify=y.

X_train, X_test, y_train, y_test = train_test_split(X_clean , y, test_size=0.20, random_state=42, stratify=y)

if X_train.empty or  X_test.empty or y_train.empty or y_test.empty:
    raise NotImplementedError("Create X_train, X_test, y_train, y_test.")

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Train shape: (7506, 25)
Test shape: (1877, 25)
Train target rate: 0.6886490807354116
Test target rate: 0.688865210442195


## 6. Build preprocessing

Use an sklearn `ColumnTransformer`.

Required preprocessing:

- numeric columns:
  - median imputation
  - standard scaling
- categorical columns:
  - most-frequent imputation
  - one-hot encoding

The logged model must be a full sklearn `Pipeline`, not just the estimator.

In [7]:
def make_one_hot_encoder():
    """Return OneHotEncoder compatible with multiple sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


# TODO: identify numeric_cols and categorical_cols from X_clean.
# Hint: numeric columns usually have dtype int/float.

numeric_cols = X_clean.select_dtypes(include=['number']).columns.tolist()     

# Everything else can be treated as categorical.
categorical_cols = X_clean.select_dtypes(exclude=['number']).columns.tolist()  


#raise NotImplementedError("Create numeric_cols and categorical_cols.")

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_cols),
        ("categorical", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))


Numeric columns: 22
Categorical columns: 3


## 7. Evaluation helpers

Complete the evaluation helper.

Every run must log the same metric set:

```text
accuracy
precision
recall
f1
roc_auc
```

Use `zero_division=0` for precision/recall/f1.

In [8]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1].astype(float)
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X).astype(float)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X).astype(float)

def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted binary classifier."""
    # TODO:
    # 1. get positive scores  
   # y_score = get_positive_scores(model, X_test)
    
    y_score = get_positive_scores(model, X_test)
    y_score = np.asarray(y_score, dtype=float)
    y_test = np.asarray(y_test, dtype=int)
    # 2. convert scores to predictions using threshold
    y_pred =  (y_score >= threshold).astype(int)
    print(y_test.dtype)
    print(np.unique(y_test))
    # 3. calculate accuracy, precision, recall, f1, roc_auc
    
    metrics = {
        "accuracy" : accuracy_score(y_test,y_pred ),
        "precision" : precision_score(y_test, y_pred, zero_division = 0) ,
        "recall" : recall_score(y_test, y_pred, zero_division = 0) ,
        "f1" : f1_score(y_test, y_pred, zero_division = 0),
        "roc_auc" : roc_auc_score(y_test, y_score)
    }
    # 4. return metrics dict, y_pred, y_score
    return metrics, y_pred, y_score
    
    
    

    #raise NotImplementedError("Complete evaluate_binary_classifier.")

## 8. Artifact helpers

Each serious run should save useful artifacts:

- confusion matrix image
- classification report JSON
- feature column list JSON
- dataset metadata snapshot JSON

Artifacts are important because MLflow should store more than scalar metrics.

In [9]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ARTIFACT_root = Path("outputs/mlflow_artifacts")
ARTIFACT_root.mkdir(parents=True, exist_ok=True)


def save_run_artifacts(run_name, y_true, y_pred, feature_cols, metadata):
    """Save local artifact files for one run and return the run artifact directory."""
    # TODO:
    # 1. create a run-specific artifact folder
    ARTIFACT_DIR = ARTIFACT_root / run_name
    print(ARTIFACT_DIR) 
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

    # 2. save confusion_matrix.png
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred)
    plt.savefig(ARTIFACT_DIR/"confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()
    
    # 3. save classification_report.json
    report = classification_report(y_true, y_pred, output_dict=True)
    
    with open(ARTIFACT_DIR/"classification_report.json", "w") as f:
        json.dump(report, f, indent=2)
    
    # 4. save feature_columns.json
    with open(ARTIFACT_DIR/"feature_columns.json", "w") as f:
        json.dump(list(feature_cols), f, indent=2)
       
    # 5. save dataset_metadata_snapshot.json
    with open(ARTIFACT_DIR/"dataset_metadata_snapshot.json", "w") as f:
        json.dump(metadata, f, indent=2)
    
    return ARTIFACT_DIR

#raise NotImplementedError("Complete save_run_artifacts.")

## 9. MLflow run helper

Complete a helper that:

1. fits the pipeline,
2. evaluates it,
3. logs params,
4. logs metrics,
5. logs tags,
6. logs artifacts,
7. logs the full sklearn Pipeline model.

Use the same helper for all model versions. That is the point of experiment tracking.

In [10]:
def run_mlflow_experiment(
    run_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    model_params,
    tags,
    threshold=0.5,
):
    # TODO: implement this function.
    
    
    # Required MLflow calls:
    # - mlflow.start_run(run_name=run_name)
    # - mlflow.log_params(...)
    # - mlflow.log_metrics(...)
    # - mlflow.set_tags(...)
    # - mlflow.log_artifacts(...)
    # - mlflow.sklearn.log_model(...)
    
    with mlflow.start_run(run_name=run_name):
        
        pipeline.fit(X_train, y_train)
        
        
        mlflow.log_params(model_params)
        mlflow.log_params({
            "threshold": threshold
            })
        
        metrics, y_pred, y_score = evaluate_binary_classifier(
            pipeline,
            X_test,
            y_test,
            threshold=threshold
        )

        mlflow.log_metrics(metrics)

        mlflow.set_tags(tags)
        
        ARTIFACT_DIR = save_run_artifacts(
            run_name,
            y_test,
            y_pred,
            feature_cols,
            metadata
            )
        
        if ARTIFACT_DIR is not None:
            mlflow.log_artifacts(ARTIFACT_DIR)
        
        mlflow.sklearn.log_model(
            pipeline,
            artifact_path="model" )
    
    


    #raise NotImplementedError("Complete run_mlflow_experiment.")

## 10. Run 0 — intentionally leaky model

This run is wrong on purpose.

Use a real model, but include `future_available_rate_30d`.

Expected behavior: performance may look suspiciously strong.

Required tags:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
model_family = logistic_regression
```

In [11]:
# TODO:
# 1. split X_leaky and y using the same stratified split settings
X_train, X_test, y_train, y_test = train_test_split(X_leaky , y, test_size=0.20, random_state=42, stratify=y)

if X_train.empty or  X_test.empty or y_train.empty or y_test.empty:
    raise NotImplementedError("Create X_leaky, X_test, y_train, y_test.")


tags = {
    "leakage_status": "leaky",
    "known_defect": "uses future_available_rate_30d",
    "model_family":"logistic_regression"
        }

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression())
])

# 2. build a LogisticRegression pipeline

run_mlflow_experiment(
    run_name = "v0_leaky_logistic_regression",
    pipeline = pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train= y_train,
    y_test=y_test,
    feature_cols = leaky_feature_cols,
    model_params={"model": "LogisticRegression"},
    tags = tags
)

# 3. log the run to MLflow

#raise NotImplementedError("Run and log v0_leaky_logistic_regression.")

int64
[0 1]
outputs\mlflow_artifacts\v0_leaky_logistic_regression


2026/06/09 04:27:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v0_leaky_logistic_regression at: http://185.50.38.163:33014/#/experiments/36/runs/7f5754d3afba4088a7f2c83651a25de8
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/36


## 11. Run 1 — dummy baseline

Train a `DummyClassifier(strategy="most_frequent")`.

This tells you what a useless model can achieve.

If your real model barely beats this, your model is weak.

In [12]:
# TODO: build and log dummy baseline.
tags = {
    "leakage_status": "leaky",
    "known_defect": "uses future_available_rate_30d",
    "model_family":"DummyClassifier(strategy='most_frequent')"
    }



run_mlflow_experiment(
    run_name = "v1_dummy_baseline",
    pipeline = DummyClassifier(strategy="most_frequent"),
    X_train=X_train,
    X_test=X_test,
    y_train= y_train,
    y_test=y_test,
    feature_cols = clean_feature_cols,
    model_params={"model": "DummyClassifier(strategy='most_frequent')"},
    tags = tags
)


#raise NotImplementedError("Run and log v1_dummy_baseline.")

int64
[0 1]
outputs\mlflow_artifacts\v1_dummy_baseline


c:\Users\lenovo\mlenv311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\lenovo\mlenv311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\lenovo\mlenv311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
2026/06/09 04:28:02 WARNING 

🏃 View run v1_dummy_baseline at: http://185.50.38.163:33014/#/experiments/36/runs/ce1aa03468fd4e359cdbb46f4511b310
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/36


## 12. Run 2 — clean logistic regression

Train your first clean real model.

Use only `X_clean`.

Required tags:

```text
leakage_status = clean
model_family = logistic_regression
```

In [13]:
# TODO: build and log clean LogisticRegression.

# TODO:
# 1. split X_clean and y using the same stratified split settings
X_clean, X_test, y_train, y_test = train_test_split(X_clean , y, test_size=0.20, random_state=42, stratify=y)

if X_train.empty or  X_test.empty or y_train.empty or y_test.empty:
    raise NotImplementedError("Create X_clean, X_test, y_train, y_test.")


tags = {
    "leakage_status": "clean",
    "model_family":"logistic_regression"
        }

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression())
])

# 2. build a LogisticRegression pipeline

run_mlflow_experiment(
    run_name = "v2_clean_logistic_regression",
    pipeline = pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train= y_train,
    y_test=y_test,
    feature_cols = clean_feature_cols,
    model_params={"model": "LogisticRegression"},
    tags = tags
)



#raise NotImplementedError("Run and log v2_clean_logistic_regression.")

int64
[0 1]
outputs\mlflow_artifacts\v2_clean_logistic_regression


2026/06/09 04:28:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v2_clean_logistic_regression at: http://185.50.38.163:33014/#/experiments/36/runs/f1918f2751c946c39214aff0346b608f
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/36


## 13. Run 3 — class-weighted logistic regression

Train logistic regression with:

```python
class_weight="balanced"
```

Compare precision and recall against the previous clean logistic model.

In [14]:
# TODO: build and log class-weighted LogisticRegression.


tags = {
    "leakage_status": "clean",
    "model_family":"logistic_regression(class_weight='balanced')"
        }

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(class_weight='balanced'))
])

# 2. build a LogisticRegression pipeline

run_mlflow_experiment(
    run_name = "v3_balanced_logistic_regression",
    pipeline = pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train= y_train,
    y_test=y_test,
    feature_cols = clean_feature_cols,
    model_params={"model": "LogisticRegression(class_weight='balanced')"},
    tags = tags
)




#raise NotImplementedError("Run and log v3_balanced_logistic_regression.")

int64
[0 1]
outputs\mlflow_artifacts\v3_balanced_logistic_regression


2026/06/09 04:28:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v3_balanced_logistic_regression at: http://185.50.38.163:33014/#/experiments/36/runs/7fa943ab34714d578c7862515d21c0c7
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/36


## 14. Run 4 — threshold tuning

Use a fitted probability model and test several decision thresholds.

Suggested thresholds:

```text
0.30, 0.40, 0.50, 0.60
```

You may log one run per threshold.

The goal is to see how precision/recall/f1 change when the threshold changes.

In [15]:
# TODO: log threshold-tuning runs.
thresholds = [0.30, 0.40, 0.50, 0.60]

for t in thresholds:

    run_mlflow_experiment(
        run_name=f"logreg_threshold_{t}",
        pipeline=pipeline,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        feature_cols=clean_feature_cols,
        model_params={
            "model": "LogisticRegression",
            "threshold": t
        },
        tags={
            "experiment_type": "threshold_sweep",
            "threshold": str(t)
        },
        threshold=t
    )

#raise NotImplementedError("Run and log threshold tuning experiments.")

int64
[0 1]
outputs\mlflow_artifacts\logreg_threshold_0.3


2026/06/09 04:28:26 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run logreg_threshold_0.3 at: http://185.50.38.163:33014/#/experiments/36/runs/4c429c3e57864f95954083ddc3b007c7
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/36
int64
[0 1]
outputs\mlflow_artifacts\logreg_threshold_0.4


2026/06/09 04:28:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run logreg_threshold_0.4 at: http://185.50.38.163:33014/#/experiments/36/runs/bdc4f7757ac94abea3b8f01c9a888b74
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/36
int64
[0 1]
outputs\mlflow_artifacts\logreg_threshold_0.5


2026/06/09 04:28:42 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run logreg_threshold_0.5 at: http://185.50.38.163:33014/#/experiments/36/runs/2a8c5a34bc6b499eaf116a3f8644d594
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/36
int64
[0 1]
outputs\mlflow_artifacts\logreg_threshold_0.6


2026/06/09 04:28:49 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run logreg_threshold_0.6 at: http://185.50.38.163:33014/#/experiments/36/runs/cd76f64e4fae45799159555905852e71
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/36


## 15. Run 5 — tree-based model

Train a `RandomForestClassifier`.

This compares a nonlinear model against logistic regression.

Log at least these parameters:

```text
n_estimators
max_depth
min_samples_leaf
class_weight
random_state
```

In [16]:
# TODO: build and log RandomForestClassifier.


tags = {
    "leakage_status": "clean",
    "model_family":"RandomForestClassifier"
        }

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier())
])

model_params = {
    "n_estimators": 100,
    "max_depth": None,
    "min_samples_leaf": 1,
    "class_weight": None,
    "random_state": 42,
}


run_mlflow_experiment(
    run_name = "v5_random_forest",
    pipeline = pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train= y_train,
    y_test=y_test,
    feature_cols = clean_feature_cols,
    model_params=model_params,
    tags = tags
)



#raise NotImplementedError("Run and log v5_random_forest.")

int64
[0 1]
outputs\mlflow_artifacts\v5_random_forest


2026/06/09 04:28:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run v5_random_forest at: http://185.50.38.163:33014/#/experiments/36/runs/19ff4c277c4f4d5cb44e76eb66b2252f
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/36


## 16. Compare MLflow runs

Use `mlflow.search_runs` to retrieve your experiment runs.

Compare at least:

```text
run name
leakage status
model family
accuracy
precision
recall
f1
roc_auc
```

Do not select a leaky run as final candidate.

In [17]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
# TODO: retrieve MLflow runs for this experiment and create a comparison table.
# extra_columns arg is available in newer versions of mlflow
df = mlflow.search_runs(
    filter_string="status = 'FINISHED'"  
)

# df=df[["tags.leakage_status",
#                    "tags.model_family",
#                    "metrics.accuracy", 
#                    "metrics.percision",
#                    "metrics.recall",
#                    "metrics.f1", 
#                    "metrics.roc_auc"]]

x = ['run_id','tags.mlflow.runName','metrics.accuracy', 'metrics.f1', 'metrics.roc_auc',
       'metrics.recall', 'metrics.precision',  'tags.model_family', 'tags.leakage_status']

df[x].sort_values("metrics.f1", ascending=False)

# raise NotImplementedError("Create runs comparison table.")

# comparison_df

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.f1,metrics.roc_auc,metrics.recall,metrics.precision,tags.model_family,tags.leakage_status
0,19ff4c277c4f4d5cb44e76eb66b2252f,v5_random_forest,0.898775,0.927647,0.953869,0.941995,0.913728,RandomForestClassifier,clean
9,3b95f5a34a1c4e46bfa1272da235d749,v5_random_forest,0.896644,0.926067,0.951062,0.939675,0.912847,RandomForestClassifier,clean
15,5cace97c16634b73b969149cbfbcd364,v2_clean_logistic_regression,0.888652,0.921162,0.920842,0.944316,0.899116,logistic_regression,clean
6,f1918f2751c946c39214aff0346b608f,v2_clean_logistic_regression,0.888652,0.921162,0.920842,0.944316,0.899116,logistic_regression,clean
8,7f5754d3afba4088a7f2c83651a25de8,v0_leaky_logistic_regression,0.888652,0.921162,0.920842,0.944316,0.899116,logistic_regression,leaky
17,2ba606e6223d4c81b49eae8ab16cc684,v0_leaky_logistic_regression,0.888652,0.921162,0.920842,0.944316,0.899116,logistic_regression,leaky
4,4c429c3e57864f95954083ddc3b007c7,logreg_threshold_0.3,0.885988,0.919245,0.920098,0.941995,0.897568,None,None
13,c6beaddf47f14ff199bb1ec8985db4fb,logreg_threshold_0.3,0.885988,0.919245,0.920098,0.941995,0.897568,None,None
12,f21da99e0f804ed694bc0e152b024adc,logreg_threshold_0.4,0.887054,0.918774,0.920098,0.927301,0.910402,None,None
3,bdc4f7757ac94abea3b8f01c9a888b74,logreg_threshold_0.4,0.887054,0.918774,0.920098,0.927301,0.910402,None,None


## 17. Select final candidate

Pick the best **clean** run.

Do not choose the leaky run.

Selection should be based on:

- f1
- roc_auc
- precision/recall tradeoff
- no leakage
- full preprocessing Pipeline logged

Write a short explanation.

In [18]:
# TODO: set BEST_RUN_ID to the selected clean run ID.
BEST_RUN_ID = '3b95f5a34a1c4e46bfa1272da235d749'

if BEST_RUN_ID is None:
    raise ValueError("Set BEST_RUN_ID to your selected clean MLflow run ID.")

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")

print("Selected best run:", BEST_RUN_ID)

Selected best run: 3b95f5a34a1c4e46bfa1272da235d749


## Final explanation

Write 3–6 sentences:

- Which run did you select?
- Why did you select it?
- Why did you reject the leaky run?
- What would you try next?

In [19]:
# TODO: replace this text.
final_explanation = """
I chose the v5_random_forest run.
I believe v5_random_forest run is the best choice since it has the highest overall F1 score and  
ROC-AUC score and both precision and recall are high. (the best overall precision–recall trade-off)
I did not choose the leaky one because i believe it does not work well in this condition because
predicts more false positives
I would like to try threshold tuning on Random Forest next 
"""

print(final_explanation)


I chose the v5_random_forest run.
I believe v5_random_forest run is the best choice since it has the highest overall F1 score and  
ROC-AUC score and both precision and recall are high. (the best overall precision–recall trade-off)
I did not choose the leaky one because i believe it does not work well in this condition because
predicts more false positives
I would like to try threshold tuning on Random Forest next 

